# 3. Crop Health — MobileNetV2
Regroup PlantVillage classes into **healthy / stressed / diseased**.

In [ ]:
!pip -q install tensorflow kaggle

In [ ]:
import os; os.environ['KAGGLE_API_TOKEN'] = 'YOUR_KAGGLE_TOKEN'
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p /content --unzip -q
import glob, shutil, os
src = glob.glob('/content/New Plant Diseases Dataset*')[0] + '/New Plant Diseases Dataset(Augmented)/train'
dst = '/content/health'
STRESSED = {'nutrient', 'deficiency', 'stressed'}  # none in PlantVillage -> all non-healthy = diseased
for cls in os.listdir(src):
    target = 'healthy' if 'healthy' in cls.lower() else 'diseased'
    os.makedirs(f'{dst}/{target}', exist_ok=True)
    for f in os.listdir(f'{src}/{cls}')[:800]:  # cap per class
        shutil.copy(f'{src}/{cls}/{f}', f'{dst}/{target}/{cls}_{f}')
print({c: len(os.listdir(f'{dst}/{c}')) for c in os.listdir(dst)})

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

IMG=(224,224); BATCH=32
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    '/content/health', image_size=IMG, batch_size=BATCH, validation_split=0.2,
    subset='training', seed=42)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    '/content/health', image_size=IMG, batch_size=BATCH, validation_split=0.2,
    subset='validation', seed=42)
classes = train_ds.class_names; print(classes)

aug = tf.keras.Sequential([layers.RandomFlip('horizontal'), layers.RandomRotation(0.1),
                           layers.Rescaling(1./127.5, offset=-1)])
base = MobileNetV2(input_shape=IMG+(3,), include_top=False, weights='imagenet'); base.trainable=False
inp = tf.keras.Input(IMG+(3,))
x = aug(inp); x = base(x, training=False); x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x); out = layers.Dense(len(classes), activation='softmax')(x)
m = models.Model(inp, out)
m.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
          loss='sparse_categorical_crossentropy', metrics=['accuracy'])
m.fit(train_ds, validation_data=val_ds, epochs=10)

In [ ]:
conv = tf.lite.TFLiteConverter.from_keras_model(m)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
open('crop_health.tflite', 'wb').write(conv.convert())